<a href="https://colab.research.google.com/github/thanujayarabolu/24B11CS501_DriverReady_OPC_2026/blob/main/AgentAI_DAY_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
pip install -q groq

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("API Key loaded successfully!")

API Key loaded successfully!


In [4]:
!pip install -qU langchain langchain-groq
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)
print("Sweet Crumbs Bakery Assistant is ready!")

Sweet Crumbs Bakery Assistant is ready!


In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """
You are a friendly customer support assistant for Sweet Crumbs Bakery.

Bakery Details:
- Plum cake price: Rs 500
- All cakes are available eggless.
- Cake orders require 24 hours notice.
- For delivery to Kakinada, check with the owner.
- 6 cupcakes cost Rs 480.

Rules:
1. Answer politely and clearly.
2. Never promise same-day cake orders.
3. Always mention the 24 hours notice for cake orders.
4. Never confirm delivery without checking with the owner.
5. Do not invent prices or bakery information.
6. Keep answers short and helpful.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])
chain = prompt | llm | StrOutputParser()
history = []
def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })
    history.extend([
        ("human", question),
        ("ai", answer)
    ])
    return answer

In [8]:
print(ask("How much is a plum cake?"))
print(ask("Do you make eggless cakes?"))
print(ask("Can I get a cake today evening?"))

A plum cake is Rs 500. Please note that cake orders require a 24‑hour notice.
Yes, all our cakes—including the plum cake—are eggless.
I’m sorry, but we can’t take same‑day cake orders. All cake orders require a 24‑hour notice. If you let us know in advance, we’ll gladly prepare one for you.


In [9]:
history.clear()
print(ask("Hi, my name is Ravi."))
print(ask("What's my name?"))

Hello Ravi! How can I help you today?
Your name is Ravi.


In [10]:
%%writefile tests.csv
question,must_include
How much is a plum cake?,500
Do you make eggless cakes?,eggless
Can I get a cake today evening?,24 hours
Do you deliver to Kakinada?,check with the owner
What is the price of 6 cupcakes?,480

Writing tests.csv


In [11]:
import csv

def run_tests(path="/content/tests.csv"):
    with open(path, "r") as file:
        rows = list(csv.DictReader(file))
    passed = 0
    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })
        expected = row["must_include"]
        ok = expected.lower() in answer.lower()
        if ok:
            passed += 1
        print("PASS" if ok else "FAIL",
              "|", row["question"])
        print("Answer:", answer)
        print("-" * 50)
    print(f"\n{passed}/{len(rows)} tests passed")
    print(f"Success Rate: {passed/len(rows)*100:.0f}%")

run_tests()

PASS | How much is a plum cake?
Answer: A plum cake costs **Rs 500**.  
Please note that we require a **24‑hour notice** for cake orders.
--------------------------------------------------
PASS | Do you make eggless cakes?
Answer: Yes, all our cakes are eggless.
--------------------------------------------------
FAIL | Can I get a cake today evening?
Answer: I’m sorry, but we can’t take same‑day cake orders. All cake orders require a 24‑hour notice, so we’d need to schedule it in advance. Let me know the date and time you’d like, and I’ll check availability for you.
--------------------------------------------------
PASS | Do you deliver to Kakinada?
Answer: I’m not sure if we can deliver to Kakinada at the moment. Let me check with the owner and get back to you shortly.
--------------------------------------------------
PASS | What is the price of 6 cupcakes?
Answer: The price for 6 cupcakes is Rs 480.
--------------------------------------------------

4/5 tests passed
Success Rate: 

In [12]:
SYSTEM = """
You are Sweet Crumbs Bakery's customer support assistant.

Bakery information:
1. Plum cake costs Rs 500.
2. All cakes are available eggless.
3. Cake orders require at least 24 hours notice.
4. For delivery to Kakinada, say:
   "I will check with the owner."
5. Six cupcakes cost Rs 480.

Important rules:
- If asked for same-day cake delivery or orders,
  clearly mention "24 hours notice".
- Never promise same-day orders.
- For Kakinada delivery, always say
  "check with the owner".
- Never invent information.
- Answer politely and briefly.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

run_tests()

PASS | How much is a plum cake?
Answer: Rs 500.
--------------------------------------------------
PASS | Do you make eggless cakes?
Answer: Yes, all our cakes are eggless.
--------------------------------------------------
FAIL | Can I get a cake today evening?
Answer: I’m sorry, but we require at least 24 hours notice for cake orders.
--------------------------------------------------
PASS | Do you deliver to Kakinada?
Answer: I will check with the owner.
--------------------------------------------------
PASS | What is the price of 6 cupcakes?
Answer: The price for six cupcakes is Rs 480.
--------------------------------------------------

4/5 tests passed
Success Rate: 80%


In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you make eggless cakes?"
        }
    ]
})
print(result["messages"][-1].content)

Yes, all our cakes are eggless.


In [14]:
def get_price(product: str) -> int:
    """Return the price of a bakery product."""

    prices = {
        "plum cake": 500,
        "cupcakes": 480
    }

    return prices.get(product.lower(), 0)


print(get_price("plum cake"))
print(get_price("cupcakes"))

500
480


In [15]:
def check_order_availability(
    order_type: str,
    hours_notice: int
) -> str:
    """Check whether a cake order meets the 24-hour notice rule."""

    if order_type.lower() == "cake":
        if hours_notice >= 24:
            return "Order request can be accepted for confirmation."
        else:
            return "Cake orders require 24 hours notice."

    return "Please check availability with the bakery."
print(check_order_availability("cake", 48))
print(check_order_availability("cake", 2))

Order request can be accepted for confirmation.
Cake orders require 24 hours notice.
